In [4]:
import cfgrib
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

# Define your wind farm location (in lat, lon)
wind_farm_lat = 38.27  # Your wind farm latitude
wind_farm_lon = -74.68  # Your wind farm longitude

# Open the GRIB2 file using cfgrib with the appropriate filter
grib_file = 'project_folder/Wind/Nor’easter/HRRR/hrrr.t00z.wrfsubhf00.grib2'
ds = cfgrib.open_dataset(grib_file, filter_by_keys={'stepType': 'instant', 'typeOfLevel': 'heightAboveGround', 'level': 10})

# Print dataset information to understand its structure
print("Dataset structure:")
print(ds)
print("\nDimensions:")
for dim in ds.dims:
    print(f"{dim}: {ds.dims[dim]}")
print("\nVariables:")
for var in ds.variables:
    print(f"{var}: {ds[var].shape}")

# Extract the U and V wind components at 10 meters above ground level
u10 = ds['u10']
v10 = ds['v10']

# Calculate wind speed
wind_speed = np.sqrt(u10**2 + v10**2)

# Get latitude and longitude as DataArrays
lat = ds.latitude
lon = ds.longitude

# Find the grid point closest to the wind farm
# First, calculate the distance to all grid points
distance = np.sqrt((lat - wind_farm_lat)**2 + (lon - wind_farm_lon)**2)
# Find the indices of the minimum distance
min_idx = np.unravel_index(distance.argmin(), distance.shape)
# Get the coordinates at this index
farm_y_idx, farm_x_idx = min_idx

# Get the wind speed at the wind farm location
wind_speed_at_farm = wind_speed.isel(y=farm_y_idx, x=farm_x_idx).values
print(f"\nWind speed at wind farm: {wind_speed_at_farm:.2f} m/s")
print(f"Farm location indices: y={farm_y_idx}, x={farm_x_idx}")
print(f"Farm coordinates: lat={lat[farm_y_idx, farm_x_idx].values}, lon={lon[farm_y_idx, farm_x_idx].values}")

# Create a smaller region to plot (around the wind farm)
# Select a region of 100x100 grid points around the farm
y_min = max(0, farm_y_idx - 50)
y_max = min(ds.dims['y'], farm_y_idx + 50)
x_min = max(0, farm_x_idx - 50)
x_max = min(ds.dims['x'], farm_x_idx + 50)

# Plot the wind field for the selected region
plt.figure(figsize=(12, 8))
plt.pcolormesh(
    lon[y_min:y_max, x_min:x_max],
    lat[y_min:y_max, x_min:x_max],
    wind_speed[y_min:y_max, x_min:x_max],
    cmap='viridis'
)
plt.colorbar(label='Wind Speed (m/s)')
plt.scatter(wind_farm_lon, wind_farm_lat, color='red', label='Wind Farm')
plt.legend()
plt.title('Wind Speed from HRRR Data (Nor\'easter)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.show()

Can't create file 'project_folder/Wind/Nor’easter/HRRR/hrrr.t00z.wrfsubhf00.grib2.da267.idx'
Traceback (most recent call last):
  File "/opt/anaconda3/envs/wind/lib/python3.11/site-packages/cfgrib/messages.py", line 539, in from_indexpath_or_filestream
    self = cls.from_fieldset(filestream, index_keys, computed_keys)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/wind/lib/python3.11/site-packages/cfgrib/messages.py", line 379, in from_fieldset
    return cls.from_fieldset_and_iteritems(fieldset, iteritems, index_keys, computed_keys)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/wind/lib/python3.11/site-packages/cfgrib/messages.py", line 392, in from_fieldset_and_iteritems
    for field_id, raw_field in iteritems:
  File "/opt/anaconda3/envs/wind/lib/python3.11/site-packages/cfgrib/messages.py", line 292, in __iter__
    for message in self.itervalues():
  File "/o

FileNotFoundError: [Errno 2] No such file or directory: 'project_folder/Wind/Nor’easter/HRRR/hrrr.t00z.wrfsubhf00.grib2'